<a href="https://colab.research.google.com/github/ayanguin/NLP-Poster-Diagnosing-Evaluation-Instability-via-Deep-Linguistic-Fingerprinting/blob/main/FinalScript.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section - Initialization

In [ ]:
pip install vllm

In [ ]:
!pip uninstall -y torchaudio torchvision

In [ ]:
!pip install torchaudio torchvision --index-url https://download.pytorch.org/whl/cu130

# Environment setup

In [ ]:
import sys
import os
import io
import re
import polars as pl
from datasets import load_dataset
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

from google.colab import drive
from google.colab import files

# Data Selection and Pre-Processing

In [ ]:
MMLU_DATASET = load_dataset("cais/mmlu", "all", split="test")
TRUTHFULQA_DATASET = load_dataset("truthfulqa/truthful_qa", "multiple_choice", split="validation")
OPINIONQA_DATASET = load_dataset("timchen0618/opinionqa", split="test")

In [ ]:
from datasets import concatenate_datasets

# 1. Define standardization functions for each dataset's unique structure
def map_mmlu(row):
    return {
        "Standard_Question": row["question"],
        "Standard_Options": row["choices"],
        "Dataset_Source": "MMLU"
    }

def map_opinionqa(row):
    return {
        "Standard_Question": row["question"],
        "Standard_Options": row["perspectives"],
        "Dataset_Source": "OpinionQA"
    }

def map_truthfulqa(row):
    # TruthfulQA nests its multiple-choice options inside the 'mc1_targets' dictionary
    return {
        "Standard_Question": row["question"],
        "Standard_Options": row["mc1_targets"]["choices"],
        "Dataset_Source": "TruthfulQA"
    }

# 2. Apply the mapping and isolate ONLY the new standard columns
cols_to_keep = ["Standard_Question", "Standard_Options", "Dataset_Source"]

mmlu_clean = MMLU_DATASET.map(map_mmlu).select_columns(cols_to_keep)
opinionqa_clean = OPINIONQA_DATASET.map(map_opinionqa).select_columns(cols_to_keep)
truthfulqa_clean = TRUTHFULQA_DATASET.map(map_truthfulqa).select_columns(cols_to_keep)

# 3. Combine them into one unified master dataset
master_dataset = concatenate_datasets([mmlu_clean, opinionqa_clean, truthfulqa_clean])

# Shuffle to ensure the model doesn't just see one benchmark at a time
master_dataset = master_dataset.shuffle(seed=42)

print(f"Combined Master Dataset successfully created!")
print(f"Total rows: {len(master_dataset)}")
print(f"Unified Columns: {master_dataset.column_names}")

In [ ]:
print(master_dataset)
print(mmlu_clean)
print(opinionqa_clean)
print(truthfulqa_clean)

In [ ]:
import polars as pl

# 1. Grab the first 3 rows of the Hugging Face dataset (returns a dictionary)
sample_data = master_dataset[:3]

# 2. Convert to Polars for pretty formatting
preview_df = pl.DataFrame(sample_data)

# 3. Expand the table width so the options list doesn't get cut off with '...'
pl.Config.set_tbl_width_chars(150)

print("=== MASTER DATASET PREVIEW ===")
print(preview_df)

In [ ]:
# Data set filter (total 900: 300 from each dataset)

from datasets import concatenate_datasets

print("Filtering and sampling 300 rows from each source...")

# 1. Filter the master dataset by the 'Dataset_Source' column we created earlier
mmlu_full = master_dataset.filter(lambda x: x["Dataset_Source"] == "MMLU")
opinionqa_full = master_dataset.filter(lambda x: x["Dataset_Source"] == "OpinionQA")
truthfulqa_full = master_dataset.filter(lambda x: x["Dataset_Source"] == "TruthfulQA")

# 2. Shuffle each dataset with a fixed seed (42) for reproducibility, then select 300 rows
sample_mmlu = mmlu_full.shuffle(seed=42).select(range(300))
sample_opinion = opinionqa_full.shuffle(seed=42).select(range(300))
sample_truthful = truthfulqa_full.shuffle(seed=42).select(range(300))

# 3. Combine them into a new balanced master dataset
balanced_master_dataset = concatenate_datasets([sample_mmlu, sample_opinion, sample_truthful])

# 4. Shuffle the combined dataset
# This ensures the model doesn't answer 300 MMLU questions in a row before seeing an OpinionQA question
balanced_master_dataset = balanced_master_dataset.shuffle(seed=42)

# 5. Verify the final shape
print("Balanced master dataset successfully created!")
print(f"Total rows: {len(balanced_master_dataset)}")
print(balanced_master_dataset)

# Model Selection and Setup

In [ ]:
QWEN = "Qwen/Qwen2.5-1.5B-Instruct"           # Need to run T4
META = "meta-llama/Llama-3.2-3B-Instruct"     # Need to run T4
HF = "HuggingFaceTB/SmolLM2-1.7B-Instruct"    # Need to run T4
GOOGLE = "google/gemma-2-2b-it"               # Need to run A100 GPU
MISTRAL = "mistralai/Mistral-7B-Instruct-v0.3"     # Need to run A100 GPU

In [ ]:
import ipywidgets as widgets

from IPython.display import display

dropdown = widgets.Dropdown(
    options=[
        ("QWEN", QWEN),
        ("META", META),
        ("HuggingFace", HF),
        ("GOOGLE", GOOGLE),
        ("MISTRAL", MISTRAL)
    ],
    description="Model:"
)

display(dropdown)


In [ ]:
# TRIGGER THIS WHEN WANT TO RUN MODEL Llama and Google model
from google.colab import userdata
from huggingface_hub import login

# Securely fetch the token from Colab Secrets
hf_token = userdata.get('HF_Token')
login(token=hf_token)

In [ ]:
selected_option = dropdown.value

class FilenoFix:
    """Wraps a stream and exposes a real OS-level fileno(),
    working around ipykernel's OutStream not supporting it."""
    def __init__(self, stream, fd):
        self._stream = stream
        self._fd = fd
    def __getattr__(self, name):
        return getattr(self._stream, name)
    def fileno(self):
        return self._fd
    def writable(self):
        return True

"""Applies the fileno() patch only if it is actually broken."""
try:
    sys.stdout.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stdout = FilenoFix(sys.stdout, 1)

try:
    sys.stderr.fileno()
except (io.UnsupportedOperation, AttributeError):
    sys.stderr = FilenoFix(sys.stderr, 2)

# 3. Load and return the model
print(f"Loading {selected_option} into GPU memory...")
if selected_option == "google/gemma-2-2b-it":
  llm = LLM(
      model=selected_option,
      dtype="bfloat16",
      max_model_len=2048,
      enforce_eager=True
  )
else:
  llm = LLM(
      model=selected_option,
      dtype="half",
      max_model_len=2048,
      enforce_eager=True
  )


# Data 2 Model

In [ ]:



# ==========================================
# PHASE 1 & 2: DATA PREP & MODEL INIT
# ==========================================
print("Loading model and tokenizer...")
#MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

# 1. Load the tokenizer to handle the chat templates
tokenizer = AutoTokenizer.from_pretrained(selected_option)     #NEED TO CHANGE THE MODEL NAME EVERY TIME BEFORE RUNNING THE CODE


print("Loading and standardizing dataset...")
#dataset = load_dataset("timchen0618/opinionqa", split="test")
dataset = balanced_master_dataset

def standardize_and_prompt(row):
    question = row.get("Standard_Question")
    options = row.get("Standard_Options")

    labels = ["A", "B", "C", "D", "E", "F"]
    formatted_options = ""
    for i, option in enumerate(options):
        if i < len(labels):
            formatted_options += f"{labels[i]}) {option}\n"

    # Base raw text
    raw_constrained = (
        f"Answer the following multiple-choice question by outputting ONLY "
        f"the single letter (A, B, C, etc.) corresponding to the correct option.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{formatted_options}\nAnswer:"
    )

    raw_unconstrained = (
        f"Read the following question and the provided options. "
        f"Take a clear stance, explain your reasoning fully in a few sentences, "
        f"and state which option you align with.\n\n"
        f"Question: {question}\n\n"
        f"Options:\n{formatted_options}"
    )

    # Wrap the raw text in the conversational dictionary format
    chat_constrained = [{"role": "user", "content": raw_constrained}]
    chat_unconstrained = [{"role": "user", "content": raw_unconstrained}]

    # Apply the Qwen Chat Template so the model knows to answer as an assistant
    # add_generation_prompt=True adds the final token that cues the AI to start speaking
    prompt_constrained = tokenizer.apply_chat_template(chat_constrained, tokenize=False, add_generation_prompt=True)
    prompt_unconstrained = tokenizer.apply_chat_template(chat_unconstrained, tokenize=False, add_generation_prompt=True)

    return {
        "Prompt_Constrained": prompt_constrained,
        "Prompt_Unconstrained": prompt_unconstrained
    }

prepared_dataset = dataset.map(standardize_and_prompt)
print("Data preparation complete!")
print(len(prepared_dataset))




In [ ]:
import re
import polars as pl
from vllm import SamplingParams

# ==========================================
# 1. CONFIGURE SAMPLING PARAMETERS
# ==========================================
# FIX: Increase max_tokens to 5 to avoid the "Whitespace Trap"
params_run1 = SamplingParams(max_tokens=5, logprobs=5, temperature=0.0)

# Run 2: Unconstrained text generation up to 256 tokens
params_run2 = SamplingParams(max_tokens=256, temperature=0.7)

# ==========================================
# 2. DUAL-RUN INFERENCE LOOP
# ==========================================
print("Starting Dual-Run Inference Loop...")
results = []

print(f"Running inference..., Total number of questions is {len(prepared_dataset)}")

# FIX: Use enumerate() to automatically generate an index number for your ID
for idx, row in enumerate(prepared_dataset):
    # Create a surrogate ID using the loop index
    q_id = f"Q_{idx}"
    #print(q_id)

    # Grab the dataset source (MMLU, OpinionQA, or TruthfulQA) if it exists
    dataset_source = row.get("Dataset_Source", "Unknown")

    # --- 1: First-Token Logprobs ---
    out_run1 = llm.generate([row["Prompt_Constrained"]], params_run1, use_tqdm=False)

    # FIX: Extract using Regex instead of .strip() to handle leading spaces
    run1_raw_text = out_run1[0].outputs[0].text
    match_run1 = re.search(r'\b([A-D])\b', run1_raw_text)
    first_token = match_run1.group(1) if match_run1 else "UNKNOWN"

    logprobs_dict = out_run1[0].outputs[0].logprobs[0]

    # --- 2: Unconstrained Text Generation ---
    out_run2 = llm.generate([row["Prompt_Unconstrained"]], params_run2, use_tqdm=False)
    generated_text = out_run2[0].outputs[0].text

    # Parse choice letter (A, B, C, or D) from generated text
    match_run2 = re.search(r'\b([A-D])\b', generated_text)
    text_answer = match_run2.group(1) if match_run2 else "Refusal/Unclear"

    # Calculate Mismatch
    is_mismatch = (first_token != text_answer)

    results.append({
        "question_id": q_id,
        "dataset_source": dataset_source,
        "first_token_answer": first_token,
        "unconstrained_text": generated_text,
        "parsed_text_answer": text_answer,
        "is_mismatch": is_mismatch,
        "top_1st_token_logprobs": str(logprobs_dict)
    })

# ==========================================
# 3. CREATE df_results DATAFRAME
# ==========================================
df_results = pl.DataFrame(results)

# Expand display width and print summary table
pl.Config.set_tbl_width_chars(100)
print("\n--- RESULTS SUMMARY ---")

# Updated to include your dataset source in the printout
print(df_results.select([
    "question_id",
    "dataset_source",
    "first_token_answer",
    "parsed_text_answer",
    "is_mismatch"
]))




In [ ]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")

In [ ]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")

In [ ]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")

In [ ]:
print(f"\nMismatch Rate: {df_results['is_mismatch'].mean():.2%}")

# Saving the llm output to csv

In [ ]:
#import os
#from google.colab import drive
#from google.colab import files


# 1. Mount Google Drive to this Colab session
#drive.mount('/content/drive')

safe_model_name = selected_option.split("/")[0]

# 2. Define the Google Drive paths
# You can change 'MyDrive' to a specific folder path if you want (e.g., 'MyDrive/Colab Notebooks/experiment_results.csv')
csv_path = f'/content/drive/MyDrive/Colab Notebooks/NLP Poster/{safe_model_name}_dataset_experiment_results.csv'


# 1. Save the Polars DataFrame to a CSV file
df_results.write_csv(csv_path)